# E000 — Episodes Index Audit & Engine-Era Filter

**Input checklist**
- Kaggle input: `kaggle/kaggriculture-episodes-index` attached to the notebook.
- Kaggle input: this `kaggriculture_v2_suite` codebase uploaded/attached as a Dataset.
- Accelerator: **None / CPU**.
- Internet: **OFF is sufficient**.
- Expected source files: any CSV/Parquet/JSON index files under `/kaggle/input/kaggriculture-episodes-index`.

**Outputs:** schema report, normalized episode catalog, current-engine subset. This notebook intentionally makes no assumptions about the official dataset schema until it inspects it.

In [ ]:
from pathlib import Path
import os,sys,json
SUITE_CANDIDATES=[Path('/kaggle/input/kaggriculture-v2-suite'),Path('/kaggle/input/kaggriculture-v2-suite/kaggriculture_v2_suite'),Path.cwd().parent,Path.cwd()]
ROOT=next((p for p in SUITE_CANDIDATES if (p/'src'/'kagv2').exists()),None)
if ROOT is None: raise FileNotFoundError('Attach/upload kaggriculture_v2_suite as a Kaggle Dataset, or run this notebook inside the repo.')
sys.path.insert(0,str(ROOT)); WORK=Path('/kaggle/working/kagv2') if Path('/kaggle/working').exists() else ROOT/'artifacts'; WORK.mkdir(parents=True,exist_ok=True)
print('ROOT=',ROOT,'WORK=',WORK)

In [ ]:
import pandas as pd
from src.kagv2.schema import audit_root,choose_index_table,normalize_index
EP_ROOT=Path('/kaggle/input/kaggriculture-episodes-index')
if not EP_ROOT.exists():
    hits=[p for p in Path('/kaggle/input').glob('*') if 'kaggriculture' in p.name.lower() and 'episode' in p.name.lower()] if Path('/kaggle/input').exists() else []
    if hits: EP_ROOT=hits[0]
print('Episodes root:',EP_ROOT)
audit=audit_root(EP_ROOT); display(audit);audit.to_csv(WORK/'episode_schema_report.csv',index=False)

In [ ]:
p,raw=choose_index_table(EP_ROOT);catalog=normalize_index(raw)
print('Selected:',p,'shape=',catalog.shape)
print(catalog.dtypes.head(50));display(catalog.head())
if 'created_at' in catalog:
    print('date range',catalog.created_at.min(),catalog.created_at.max())
    catalog['post_town_rebalance']=catalog.created_at>=pd.Timestamp('2026-08-07',tz='UTC')
    print(catalog.post_town_rebalance.value_counts(dropna=False))
try: catalog.to_parquet(WORK/'episode_catalog.parquet',index=False)
except Exception as e: catalog.to_csv(WORK/'episode_catalog.csv',index=False); print('parquet fallback',e)

## What to inspect before continuing
The index may be one row per episode, per agent, or may contain nested agent metadata. Check the inferred ID/reward/rating columns above. If the schema is unexpectedly nested, save the schema report and use it to add one small adapter in `src/kagv2/schema.py` rather than writing notebook-specific parsing.